# **Pytorch Basics**

# Introduction

`Pytorch` is a Python package that is used to design and implement neural networks in an abstract and efficient manner. There are two main phases when it comes to implementing a neural network model for a specific classification task:

1. **Network Design** - Designing the network's architecture by specifying the number of layers, the number of neurons in each layer, the activation functions of those layers, etc.

2. **Training & Evaluation** - Allowing the model to learn the optimal network parameters through a set of labelled training data before being evaluated on it performance with respect to a different set of labelled testing data. Here is a simple psuedocode to represent the training algorithm for a network:

```
1. Initialize the weights & biases to random values.
2. For each training data point:
    a. Run the input features through the current network.
    b. Calculate the loss by passing in the output into the loss function.
    c. Perform backpropagation to compute the gradiets with respect to each parameter.
    d. Update each parameter using its computed gradient.
```

We will start by importing the necessary packages and libraries for this tutorial.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import random

# Network Design

## **Building the Network Class**

Firstly, we design the network by creating a class that inherits from the `nn.Module` class, which allows us to declare layers, activation functions, and customize the network's design. We will start by creating a simple network:

<br>

1. **Input Layer** - Three Input Neurons.
2. **Hidden Layer 1** - Four Neurons With Sigmoid Activation.
3. **Hidden Layer 2** - Three Neurons With ReLU Activation.
4. **Output Layer** - Two Neurons With Sigmoid & Softmax Activations.

<br>

The layers must be initialized in the constructor of our custom class. After that, we must also define the forward pass function using the layers we instantiated in the constructor. The advantage of using `Pytorch` is that we can customize the forward pass by adding loops for certain layers, conditional layers, etc.

In [18]:
torch.manual_seed(42)

class SimpleNetwork(nn.Module):
    def __init__(self):
        super(SimpleNetwork, self).__init__()

        # Initialize Layers (The output layer is implemented by applying the sigmoid and softmax activations to the last hidden layer.)
        self.input_layer = nn.Linear(3, 4)
        self.hidden_layer_1 = nn.Linear(4, 3)
        self.hidden_layer_2 = nn.Linear(3, 2)

        # Initialize Activation Functions
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        # Process the input through each layer and activation function.
        x = self.input_layer(x)
        x = self.sigmoid(x)
        x = self.hidden_layer_1(x)
        x = self.relu(x)
        x = self.hidden_layer_2(x)
        x = self.sigmoid(x)
        x = self.softmax(x)
        return x

# Instantiate the network model.
model = SimpleNetwork()

## **Processing an Input**

Now that we have a model, we must look at how to use the model to make predictions for a set of input data. In order to study this, we will be using the Student Depression dataset to predict whether or not a student is depressed:

In [19]:
# Import data from .csv file.
depression_data = pd.read_csv('student_depression_dataset.csv')

# Drop all rows with NaN values for the indicated columns.
depression_data = depression_data.dropna(
    subset=[
        'Academic Pressure',
        'Study Satisfaction',
        'Financial Stress',
        'Depression'
    ]
)

# Remove all rows with unknown values in each of the columns indicated below.
depression_data = depression_data[depression_data['Academic Pressure'] != '?']
depression_data = depression_data[depression_data['Study Satisfaction'] != '?']
depression_data = depression_data[depression_data['Financial Stress'] != '?']
depression_data = depression_data[depression_data['Depression'] != '?']

# Display the clean data frame.
depression_data

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,'5-6 hours',Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,'5-6 hours',Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,'Less than 5 hours',Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,'7-8 hours',Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,'5-6 hours',Moderate,M.Tech,Yes,1.0,1.0,No,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27896,140685,Female,27.0,Surat,Student,5.0,0.0,5.75,5.0,0.0,'5-6 hours',Unhealthy,'Class 12',Yes,7.0,1.0,Yes,0
27897,140686,Male,27.0,Ludhiana,Student,2.0,0.0,9.40,3.0,0.0,'Less than 5 hours',Healthy,MSc,No,0.0,3.0,Yes,0
27898,140689,Male,31.0,Faridabad,Student,3.0,0.0,6.61,4.0,0.0,'5-6 hours',Unhealthy,MD,No,12.0,2.0,No,0
27899,140690,Female,18.0,Ludhiana,Student,5.0,0.0,6.88,2.0,0.0,'Less than 5 hours',Healthy,'Class 12',Yes,10.0,5.0,No,1


We will be using the following three input features to make predictions:

<br>

1. **Academic Pressure**: A student's stress level, with 1 being the least level of stress and 5 being the highest level of stress.

2. **Study Satisfaction**: The amount of satisfaction a student feels with their academics, with 1 being the least level of satisfaction and 5 being the highest level of satisfaction.

3. **Financial Stress**: The stress level a student feels with respect to their finances, with 1 being the least level of stress and 5 being the highest level of stress.

<br>

We will be picking five random data points and using the network to classify whether or not each student is depressed. We will start by separating the inputs and outputs of each data point.

**Note:** Classification tasks like these, where the output consists of only two possibilities (yes or no), are called **Binary Classification Tasks**.

In [20]:
def perform_sampling(sample_size=None):
  depression_array = np.array(depression_data)

  if sample_size != None:
    depression_array = random.sample(list(depression_array), k=sample_size)

  inputs = []
  outputs = []

  for i in range(len(depression_array)):
      current_input = np.array([
          float(depression_array[i][5]),
          float(depression_array[i][8]),
          float(depression_array[i][15])
      ])
      current_output = int(depression_array[i][17])

      if current_output:
        current_output = [1, 0]
      else:
        current_output = [0, 1]

      inputs.append(current_input)
      outputs.append(current_output)

  inputs = np.array(inputs)
  outputs = np.array(outputs)

  return inputs, outputs

Now that we have split the inputs and outputs, we can proceed by passing in our inputs into the neural network stored by the `model` variable. When we pass an input array of type `list` or of type `np.array` into a `Pytorch` network, we must first convert it into an **Input Tensor**. An Input Tensor allows the network to accept the inputs and process each input data to return an **Output Tensor**.

In [22]:
# Randomly sample 5 data points from the data frame.
inputs, outputs = perform_sampling(sample_size=5)

# Create the input tensor.
X = torch.tensor(inputs, dtype=torch.float32)

# Run the input data through the network and store the output tensor.
predictions = model(X)

# Print the actual and expected outputs.
print('Actual Outputs:', outputs)
print('Predicted Outputs:', predictions)

Actual Outputs: [[0 1]
 [0 1]
 [1 0]
 [1 0]
 [0 1]]
Predicted Outputs: tensor([[0.5193, 0.4807],
        [0.5188, 0.4812],
        [0.5181, 0.4819],
        [0.5193, 0.4807],
        [0.5183, 0.4817]], grad_fn=<SoftmaxBackward0>)


# Training & Evaluation

As indicated in the introduction for this notebook, the training of a neural network involves four main phases for each input:

<br>

1. **Forward Propagation**: Process the input through the network.

2. **Loss Computation**: Compute the loss given the network's predictions and the expected output.

3. **Backpropagation**: Compute the gradient of the loss function with respect to each parameter of the network $\frac{∂C}{∂P}$ where $P$ represents a parameter of the network.

4. **Parameter Update**: Update each network parameter with respect to its gradient and the **learning rate** $α$:

  $w^{(i)}_{j}=w^{(i)}_{j}-\alpha\frac{∂C}{∂w^{(i)}_{j}}$

  $b^{(i)}_{j}=b^{(i)}_{j}-\alpha\frac{∂C}{∂b^{(i)}_{j}}$

  Where $w^{(i)}_{j}$ and $b^{(i)}_{j}$ represent the $j^{th}$ weight and bias of the $i^{th}$ layer, respectively.

<br>

The **Learning Rate** is allows each parameter to take gradual steps towards the local minimum of the loss function. This is a hyper-parameter that is set to a constant value between $[0,1]$ before the training starts. If the learning rate is too low, the step size towards the local minimum may be too small, so the model may not even converge to the local minimum. If the learning rate is too high, the model may "jump around" the loss function and may not converge to the local minimum either. As a result, a good learning rate must be tuned.

<br>

Luckily, each of the four phases above are implemented in `Pytorch`, and we just need to call the respective functions for each of the phases. We will use approximately 70% of our data to train our network and the remaining 30% of our data to evaluate our network.

In [24]:
# Separate the input features and the expected output for each data point.
inputs, outputs = perform_sampling()

# Allocate the training data.
train_inputs = inputs[0:int(0.7*len(inputs))]
train_outputs = outputs[0:int(0.7*len(outputs))]

# Allocate the evaluation data.
test_inputs = inputs[int(0.7*len(inputs)):len(inputs)]
test_outputs = outputs[int(0.7*len(outputs)):len(outputs)]

Now that we have our training and testing data separated, we can start training the network with our training data. we will be using the learning rate $\alpha=0.01$, with a **Cross-Entropy Loss Function** and a **Stochastic Gradient Descent** minimizing technique.

In [25]:
# Declare the learning rate, loss function, and the optimizer.
learning_rate = 0.01
loss = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)
epoch = 100

# Train the network for just 1 epoch.
X = torch.tensor(train_inputs, dtype=torch.float32)
y = torch.tensor(train_outputs, dtype=torch.float32)

for i in range(epoch):
  predictions = model(X)  # Forward Propagation
  loss_value = loss(predictions, y) # Loss Computation
  loss_value.backward() # Backpropagation
  optimizer.step()  # Parameter Update

Now that we have trained the network, we can evaluate the accuracy of the network. In order to classify whether a student is depressed, we will take the prediction of the network that the student is depressed and we will round it to the nearest integer.

In [26]:
X = torch.tensor(test_inputs, dtype=torch.float32)
y = np.array(test_outputs)

predictions = model(X)
predictions = predictions.detach().numpy()

correct_count = 0

for i in range(len(predictions)):
  if np.argmax(predictions[i]) == np.argmax(y[i]):
    correct_count += 1

print("Accuracy: " + str((correct_count * 100)/len(predictions)) + "%")

Accuracy: 58.5663082437276%
